# Model Evaluation


In [1]:
from types import SimpleNamespace
import os
import sys
import argparse
from IPython import get_ipython
import torch
import lightning as pl

from meta import update_meta

from data_module import NPZTensorDataModule
from model import MultiLayerLeakyReLUModel

detect if runnig as notebook or script


In [2]:
if get_ipython():
    is_running_in_notebook = True
    print("executing as notebook ...")
else:
    is_running_in_notebook = False
    print("executing as script ...")

executing as notebook ...


## Args


### Demo


In [3]:
# args = SimpleNamespace(
#     # io
#     data_path="data/demo",
#     logs_path="data/logs",
#     experiment_name="demo_experiment",
#     version="v1",
#     # data
#     batch_size=64,
#     num_workers=4,
#     persistent_workers=True,
#     pin_memory=True,
# )

### laser-pulse-shaping-astra-sim


In [4]:
args = SimpleNamespace(
    # io
    data_path="data/laser-pulse-shaping-astra-sim-v11-normalized",
    logs_path="data/logs",
    experiment_name="laser-pulse-shaping-astra-sim-v11-normalized",
    version="v1",
    # data
    batch_size=64,
    num_workers=4,
    persistent_workers=True,
    pin_memory=True,
)

In [5]:
if not is_running_in_notebook:
    parser = argparse.ArgumentParser()
    # io
    parser.add_argument("--data_path", type=str, required=True)
    parser.add_argument("--logs_path", type=str, required=True)
    parser.add_argument("--experiment_name", type=str, required=True)
    parser.add_argument("--version", type=str, required=True)
    # data
    parser.add_argument("--batch_size", type=int, required=True)
    parser.add_argument("--num_workers", type=int, required=True)
    parser.add_argument("--persistent_workers", type=bool, required=True)
    parser.add_argument("--pin_memory", type=bool, required=True)

    args = parser.parse_args(sys.argv[1:])

In [6]:
version_path = os.path.join(args.logs_path, args.experiment_name, args.version)
update_meta(version_path, vars(args))

In [7]:
evaluation_path = os.path.join(version_path, "eval")
os.makedirs(evaluation_path, exist_ok=True)

In [8]:
ckpt_path = os.path.join(version_path, "best.ckpt")

## load data


In [9]:
data = NPZTensorDataModule(
    path=args.data_path,
    batch_size=args.batch_size,
    num_workers=args.num_workers,
    persistent_workers=args.persistent_workers,
    pin_memory=args.pin_memory,
)
data.setup("fit")

## store meta


In [10]:
ckpt = torch.load(ckpt_path)
meta = dict(best_epoch=ckpt["epoch"], best_step=ckpt["global_step"])
del ckpt
update_meta(version_path, meta)

## load model


In [11]:
model = MultiLayerLeakyReLUModel.load_from_checkpoint(ckpt_path)

## Evaluate Best Model


### validation dataset


In [12]:
trainer = pl.Trainer(logger=False)

/data/dust/user/kwasniok/astra/opal-photo-injector-surrogate-model/.venv/lib/python3.11/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /data/dust/user/kwasniok/astra/opal-photo-injector-s ...
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


In [13]:
best_val_results = trainer.validate(model, data.val_dataloader())[0]

meta = {f"best_{k.replace('/', '_')}": v for k, v in best_val_results.items()}
update_meta(version_path, meta)

/data/dust/user/kwasniok/astra/opal-photo-injector-surrogate-model/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Validation: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃      Validate metric      ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         val/loss          │    0.1826925128698349     │
└───────────────────────────┴───────────────────────────┘

### training dataset


In [14]:
best_train_results = trainer.validate(model, data.train_dataloader())[0]

meta = {
    f"best_{k.replace('val/', 'train/').replace('/', '_')}": v
    for k, v in best_train_results.items()
}
update_meta(version_path, meta)

/data/dust/user/kwasniok/astra/opal-photo-injector-surrogate-model/.venv/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:476: Your `val_dataloader`'s sampler has shuffling enabled, it is strongly recommended that you turn shuffling off for val/test dataloaders.


Validation: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃      Validate metric      ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         val/loss          │    0.17831586301326752    │
└───────────────────────────┴───────────────────────────┘